# Validation Notebook to compare to jlbernal/lim 

## Initial Setup

In [ ]:
import os
import sys

sys.path.append("../")
envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
from copy import copy, deepcopy
import numpy as np
import seaborn
from getdist.gaussian_mixtures import GaussianND
from getdist import plots
from scipy.optimize import curve_fit

In [ ]:
# seaborn.set_theme(rc={'axes.edgecolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black',})
import niceplots.utils as nicepl
nicepl.initPlot()

In [ ]:
Cs = seaborn.color_palette("colorblind")
Cp = seaborn.color_palette("Paired")
Cs

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.cosmology import cosmology
from SSLimPy.cosmology import halo_model
from SSLimPy.cosmology import astro
from SSLimPy.LIMsurvey import covariance as scov
import SSLimPy.LIMsurvey.power_spectrum as spobs
from SSLimPy.utils.utils import *

In [ ]:
sys.path.append("../../lim")
import lim
import source.mass_luminosity as ml

## Choose model parameters and save them in dictionaries

In [ ]:
settings = {
    "code":"class", # The Einstein--Boltzman solver that should be used
    "do_RSD" : True, # If RSD should be considerd
    "nonlinearRSD" : True, # If you want to add FOG to the RSD
    "QNLpowerspectrum": False, # Use dewiggled power spectrum (vlasov approximation of nonlinear structure formation)
    "FoG_damp" : "ISTF_like", # The particular parametrization for the FOG. Check PowerSpectrum for the full list
    "halo_model_PS" : True, # If the cosmological shotnoise should be computed from the halo model 
    "output" : ["Power spectrum", "Covariance"], # What output one wants (here power spectrum and Gaussian covariance only)
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
}

cosmodict={
    "h": 0.677,
    "Omegam": 0.309167,
    "Omegab": 0.04903,
    "sigma8":0.8222,
    "ns":0.96824,
    "mnu":0.06,
    "Neff":3.044,
}

# Parameters that enter your halo model. Typically they are not changed but you could
halodict={
    "halo_tracer" : "clustering", # Computes all halo quantities from the matter field - neutrinos
    "hmf_model": "ST", # Sheth--Tormann halo mass function
    "concentration": "Diemer19", # Diemer19 halo concentration relation
    "bias_model": "ST99",
}

In [ ]:
# Parameters for the Survey specifications
def get_specs(nu):
        z = 1
        nuObs = nu / (z + 1)
        DeltaDeltanu = 0.1

        surveyspecs = {
                "Tsys_NEFD": 0 * u.uK, #System temperature for instrumental shotnoise
                "Nfeeds": 19,
                "tobs": 1300 * u.h,
                "nD": 1, # Observational parameters
                "beam_FWHM": 0. * u.arcmin,
                "nu":  nu,
                "nuObs": nuObs, # Observed Frequency
                "Delta_nu": DeltaDeltanu * nuObs, # Frequency Bin
                "dnu": 0. * u.MHz, # Spectrograph resolution
                "Omega_field": 140 * u.deg**2, # Angular size of survey
        }
        return surveyspecs

# CO21
SSlimpy part

In [ ]:
# Parameters for Astrophysics.
astrodict_CO21={
    "model_type": "ML",
    "model_name": "TonyLi",
    "model_par": {
        "alpha": 1.11,
        "beta": 0.6,
        "dMF": 1 * u.Msun * u.yr**-1 * u.Lsun**-1,
        "sig_SFR":0,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37
} # CO21

nu = 2 * 115.27 * u.GHz # CO(2-1)
surveyspecs_CO21 = get_specs(nu)

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict_CO21,
    obspars_dict=surveyspecs_CO21,
)

In [ ]:
myastro = myssl.current_astro
mycosmo = myastro.cosmology
h = mycosmo.h()

Create lim instance

In [ ]:
m = lim.lim(
    {
        "cosmo_code": "class",
        "cosmo_input_class" : mycosmo.classcosmopars,
        "model_type": "ML",
        "model_name": "TonyLi",
        "model_par": {
            "alpha": 1.11,
            "beta": 0.6,
            "dMF": 1 * u.Msun * u.yr**-1 * u.Lsun**-1,
            "sig_SFR":0,
            "SFR_file": "sfr_release.dat",
            "do_quench": False,
        },
        "sigma_scatter" : 0.37,
        "hmf_model": "ST", # Sheth--Tormann halo mass function
        "bias_model": "Tinker10",
        "nu":  surveyspecs_CO21["nu"],
        "dnu": 0. * u.MHz, # Spectrograph resolution
        "nuObs": surveyspecs_CO21["nuObs"], # Observed Frequency
        "Delta_nu": surveyspecs_CO21["Delta_nu"],
        "do_onehalo":True,
    }
)

In [ ]:
mask = np.logical_and((m.M > 1e9 * u.Msun), (m.M < 1e14 * u.Msun))
M = m.M[mask]

In [ ]:
lLM = m.LofM[mask]
sLM = myastro.massluminosityfunction(M, 1.0)

plt.loglog(M, lLM)
plt.loglog(M, sLM)

In [ ]:
plt.loglog(M, myastro.halomodel.halomassfunction(M, 1.0) /m.dndM[mask])

In [ ]:
print(myastro.Tavg(1.0, 1), m.Tmean)
print(myastro.Tavg(1.0, 2), m.Pshot)
print(myastro.bavg(1, 1, 1), m.bavg)

In [ ]:
pobs = spobs.PowerSpectra(myastro)
pobs_lim = m.Pk_0

plt.loglog(m.k, pobs_lim.to(u.Mpc**3 * u.uK**2), c=Cp[0], label="lim")
plt.loglog(pobs.k, pobs.Pk_0bs, c=Cp[1], ls="--", label="SSLimPy")
plt.legend()
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mu K^2\,\mathrm{Mpc}^{3}]$")
plt.title("CO2-1")

# C[II]

In [ ]:
astrodict_CII={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37
}

nu = 1.897 * u.THz
surveyspecs_CII = get_specs(nu)

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict_CII,
    obspars_dict=surveyspecs_CII,
)

myastro = myssl.current_astro
mycosmo = myastro.cosmology
h = mycosmo.h()

In [ ]:
m = lim.lim(
    {
        "cosmo_code": "class",
        "cosmo_input_class" : mycosmo.classcosmopars,
        "model_type": "ML",
        "model_name": "SilvaCII",
        "model_par": {
            "a": 0.8475,
            "b": 7.2203,
            "SFR_file": "sfr_release.dat",
            "do_quench": False,
        },
        "sigma_scatter" : 0.37,
        "hmf_model": "ST", # Sheth--Tormann halo mass function
        "bias_model": "Tinker10",
        "nu":  surveyspecs_CII["nu"],
        "dnu": 0. * u.MHz, # Spectrograph resolution
        "nuObs": surveyspecs_CII["nuObs"], # Observed Frequency
        "Delta_nu": surveyspecs_CII["Delta_nu"],
        "do_onehalo":True,
    }
)

In [ ]:
lLM = m.LofM[mask]
sLM = myastro.massluminosityfunction(M, 1.0)

plt.loglog(M, lLM)
plt.loglog(M, sLM)

In [ ]:
print(myastro.Tavg(1.0, 1), m.Tmean)
print(myastro.Tavg(1.0, 2), m.Pshot)
print(myastro.bavg(1, 1, 1), m.bavg)

In [ ]:
pobs = spobs.PowerSpectra(myastro)
pobs_lim = m.Pk_0

plt.loglog(m.k, pobs_lim.to(u.Mpc**3 * u.uK**2), c=Cp[0], label="lim")
plt.loglog(pobs.k, pobs.Pk_0bs, c=Cp[1], ls="--", label="SSLimPy")
plt.legend()
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mu K^2\,\mathrm{Mpc}^{3}]$")
plt.title("C[II]")